# UrbanFloodBench: Coupled 1D + 2D Overnight Training (Re-release)

This notebook trains a **coupled** spatiotemporal model that predicts **both 1D and 2D water levels**, with explicit **1D↔2D link edges** for surcharge exchange.

- **2D data**: unchanged in the re-release
- **1D dynamic data**: fixed in the re-release (node labels were previously misassigned)

This notebook:
1) builds strict `node_idx -> row` mappings from **static** node lists  
2) caches **memmaps** for fast training  
3) trains a coupled **MPNN + GRU** with typed message passing (1D edges, 2D edges, link edges)  
4) checkpoints so you can resume if anything explodes (humans love that)

> You are deleting the whole dataset and caches. Great. This notebook recreates caches from scratch.


In [1]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

DATA_ROOT = "data"
MODELS = ["Model_1", "Model_2"]

SPLIT_TRAIN = "train"
SPLIT_TEST  = "test"

T = 445          # timesteps per event
WARMUP = 9       # timesteps with full history (teacher forcing)

# Overnight settings (tune if needed)
STEPS = 20000    # per model
L = 32           # rollout window length
ROUNDS = 2       # message passing rounds per step

# Subgraph sampling (this is the difference between "overnight" and "all weekend")
SEED_N2 = 2048
NEIGH_K2 = 8
SEED_N1 = 1024
NEIGH_K1 = 8
EXPAND_1D_HOPS = 1

# Optimization
LR = 2e-4
WD = 1e-4
CLIP = 1.0

# Joint loss weight
ALPHA_1D = 1.0

# Scheduled sampling
SS_WARM = 3000
SS_MAXP = 0.6

# Checkpoints
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Optional: warm-start 2D branch from a saved 2D-only checkpoint you kept elsewhere.
# Put it here: checkpoints/Model_1_2d_ckpt.pt and checkpoints/Model_2_2d_ckpt.pt
USE_PRETRAIN_2D = True

# Reproducibility
SEED = 6
rng = np.random.RandomState(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (device.type == "cuda")
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
USE_SCALER = USE_AMP and (AMP_DTYPE == torch.float16)

print("device:", device)
print("amp:", USE_AMP, "dtype:", AMP_DTYPE, "scaler:", USE_SCALER)

device: cuda
amp: True dtype: torch.bfloat16 scaler: False


In [2]:
def ss_prob(step, warm=SS_WARM, maxp=SS_MAXP):
    p = min(1.0, step / warm)
    return float(maxp * p)

def list_events(model_name, split):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    evs = sorted([d for d in os.listdir(base) if d.startswith("event_") and os.path.isdir(os.path.join(base, d))])
    return base, evs

def ensure_cache_dir(model_name, split):
    cache_dir = f"{DATA_ROOT}/{model_name}/{split}/_cache_rerelease_v1"
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir

def save_ckpt(path, model, opt, step):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step}, path)

def load_ckpt(path, model, opt):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    return int(ckpt["step"])

def _pick_conn_cols(df):
    candidates = [
        ("node_idx_1d", "node_idx_2d"),
        ("node_1d", "node_2d"),
        ("idx_1d", "idx_2d"),
        ("node1d", "node2d"),
    ]
    cols = set(df.columns)
    for a, b in candidates:
        if a in cols and b in cols:
            return a, b
    return df.columns[-2], df.columns[-1]


In [3]:
def load_static(model_name):
    train_dir = f"{DATA_ROOT}/{model_name}/{SPLIT_TRAIN}"

    df_1d_edge_index    = pd.read_csv(f"{train_dir}/1d_edge_index.csv")
    df_1d_edges_static  = pd.read_csv(f"{train_dir}/1d_edges_static.csv")
    df_1d_nodes_static  = pd.read_csv(f"{train_dir}/1d_nodes_static.csv")
    df_1d2d_connections = pd.read_csv(f"{train_dir}/1d2d_connections.csv")

    df_2d_edge_index    = pd.read_csv(f"{train_dir}/2d_edge_index.csv")
    df_2d_edges_static  = pd.read_csv(f"{train_dir}/2d_edges_static.csv")
    df_2d_nodes_static  = pd.read_csv(f"{train_dir}/2d_nodes_static.csv")

    if "area" in df_2d_nodes_static.columns:
        df_2d_nodes_static["area"] = df_2d_nodes_static["area"].clip(lower=0)

    if "min_elevation" in df_2d_nodes_static.columns and "elevation" in df_2d_nodes_static.columns:
        mask_nan = df_2d_nodes_static["min_elevation"].isna()
        df_2d_nodes_static.loc[mask_nan, "min_elevation"] = df_2d_nodes_static.loc[mask_nan, "elevation"]
        mask = df_2d_nodes_static["min_elevation"] > df_2d_nodes_static["elevation"]
        df_2d_nodes_static.loc[mask, "min_elevation"] = df_2d_nodes_static.loc[mask, "elevation"]

    return (df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
            df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static)


In [4]:
def build_pack(df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
               df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static):

    node_ids_1d = df_1d_nodes_static["node_idx"].to_numpy(np.int64)
    node_ids_2d = df_2d_nodes_static["node_idx"].to_numpy(np.int64)
    N1 = node_ids_1d.shape[0]
    N2 = node_ids_2d.shape[0]

    id2i_1d = {int(n): i for i, n in enumerate(node_ids_1d)}
    id2i_2d = {int(n): i for i, n in enumerate(node_ids_2d)}

    # 1D edges (assume src/dst in last 2 cols)
    ei1 = df_1d_edge_index.to_numpy()
    src1 = ei1[:, -2].astype(np.int64)
    dst1 = ei1[:, -1].astype(np.int64)
    src1 = np.array([id2i_1d[int(x)] for x in src1], dtype=np.int64)
    dst1 = np.array([id2i_1d[int(x)] for x in dst1], dtype=np.int64)

    # 2D edges (assume src/dst in last 2 cols)
    ei2 = df_2d_edge_index.to_numpy()
    src2 = ei2[:, -2].astype(np.int64)
    dst2 = ei2[:, -1].astype(np.int64)
    src2 = np.array([id2i_2d[int(x)] for x in src2], dtype=np.int64)
    dst2 = np.array([id2i_2d[int(x)] for x in dst2], dtype=np.int64)

    # link edges (1D<->2D)
    c1, c2 = _pick_conn_cols(df_1d2d_connections)
    conn_1d_ids = df_1d2d_connections[c1].to_numpy(np.int64)
    conn_2d_ids = df_1d2d_connections[c2].to_numpy(np.int64)
    conn_src1 = np.array([id2i_1d[int(x)] for x in conn_1d_ids], dtype=np.int64)
    conn_dst2 = np.array([id2i_2d[int(x)] for x in conn_2d_ids], dtype=np.int64)

    def _norm_df(df, drop_cols):
        x = df.select_dtypes(include=[np.number]).copy()
        for c in drop_cols:
            if c in x.columns:
                x = x.drop(columns=[c])
        arr = x.to_numpy(np.float32)
        mu = arr.mean(axis=0, keepdims=True)
        sd = arr.std(axis=0, keepdims=True)
        sd[sd < 1e-6] = 1e-6
        arr = (arr - mu) / sd
        return arr

    node_1d_np = _norm_df(df_1d_nodes_static, ["node_idx"])
    edge_1d_np = _norm_df(df_1d_edges_static, ["edge_idx"])

    node_2d_np = _norm_df(df_2d_nodes_static, ["node_idx"])
    edge_2d_np = _norm_df(df_2d_edges_static, ["edge_idx"])

    link_num = df_1d2d_connections.select_dtypes(include=[np.number]).copy()
    for c in [c1, c2, "node_idx_1d", "node_idx_2d", "node_idx"]:
        if c in link_num.columns:
            link_num = link_num.drop(columns=[c])
    if link_num.shape[1] == 0:
        link_np = np.zeros((df_1d2d_connections.shape[0], 1), dtype=np.float32)
    else:
        link_np = link_num.to_numpy(np.float32)
        mu = link_np.mean(axis=0, keepdims=True)
        sd = link_np.std(axis=0, keepdims=True)
        sd[sd < 1e-6] = 1e-6
        link_np = (link_np - mu) / sd

    # torch tensors
    node_static_1d = torch.tensor(node_1d_np, device=device)
    edge_attr_1d   = torch.tensor(edge_1d_np, device=device)
    node_static_2d = torch.tensor(node_2d_np, device=device)
    edge_attr_2d   = torch.tensor(edge_2d_np, device=device)
    link_attr      = torch.tensor(link_np, device=device)

    # CSR for 1D sampling
    order1 = np.argsort(src1)
    src1_s = src1[order1]
    dst1_s = dst1[order1]
    indptr1 = np.zeros(N1 + 1, dtype=np.int64)
    np.add.at(indptr1, src1_s + 1, 1)
    indptr1 = np.cumsum(indptr1)

    # CSR for 2D sampling
    order2 = np.argsort(src2)
    src2_s = src2[order2]
    dst2_s = dst2[order2]
    indptr2 = np.zeros(N2 + 1, dtype=np.int64)
    np.add.at(indptr2, src2_s + 1, 1)
    indptr2 = np.cumsum(indptr2)

    return {
        "N1": N1, "N2": N2,
        "id2i_1d": id2i_1d, "id2i_2d": id2i_2d,
        "node_ids_1d": node_ids_1d, "node_ids_2d": node_ids_2d,

        "src1": src1, "dst1": dst1,
        "src2": src2, "dst2": dst2,

        "src1_s": src1_s, "dst1_s": dst1_s, "indptr1": indptr1,
        "src2_s": src2_s, "dst2_s": dst2_s, "indptr2": indptr2,

        "conn_src1": conn_src1, "conn_dst2": conn_dst2,
        "link_attr": link_attr,

        "node_static_1d": node_static_1d,
        "edge_attr_1d": edge_attr_1d,
        "node_static_2d": node_static_2d,
        "edge_attr_2d": edge_attr_2d,
    }


In [ ]:
def sanity_check_1d_dynamic(model_name, split, pack, n_events=3):
    base, evs = list_events(model_name, split)
    pick = evs[:min(n_events, len(evs))]
    static_ids = set(pack["node_ids_1d"].astype(int).tolist())

    for ev in pick:
        f = f"{base}/{ev}/1d_nodes_dynamic_all.csv"
        df = pd.read_csv(f, usecols=["node_idx", "timestep", "water_level"])
        dyn_ids = set(df["node_idx"].astype(int).unique().tolist())

        extra = dyn_ids - static_ids
        missing = static_ids - dyn_ids

        print(model_name, split, ev,
              "static_nodes=", len(static_ids),
              "dyn_nodes=", len(dyn_ids),
              "extra=", len(extra),
              "missing=", len(missing))

        if len(extra) > 0:
            print("  example extra ids:", sorted(list(extra))[:10])

def sanity_check_2d_dynamic(model_name, split, pack, n_events=2):
    base, evs = list_events(model_name, split)
    pick = evs[:min(n_events, len(evs))]
    static_ids = set(pack["node_ids_2d"].astype(int).tolist())

    for ev in pick:
        f = f"{base}/{ev}/2d_nodes_dynamic_all.csv"
        df = pd.read_csv(f, usecols=["node_idx", "timestep", "water_level", "rainfall"])
        dyn_ids = set(df["node_idx"].astype(int).unique().tolist())

        extra = dyn_ids - static_ids
        missing = static_ids - dyn_ids

        print(model_name, split, ev,
              "static_nodes=", len(static_ids),
              "dyn_nodes=", len(dyn_ids),
              "extra=", len(extra),
              "missing=", len(missing))

        if len(extra) > 0:
            print("  example extra ids:", sorted(list(extra))[:10])


In [6]:
# -----------------------------
# Cache dynamic CSVs into memmaps
#   2D: (T, N2, 2) -> [water_level, rainfall]
#   1D: (T, N1, 1) -> [water_level]
# -----------------------------
def cache_event_2d(model_name, split, ev, N2, id2i_2d):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    cache_dir = ensure_cache_dir(model_name, split)
    out_dir = f"{cache_dir}/{ev}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/2d_dyn.dat"
    if os.path.exists(out_path):
        return

    mm = np.memmap(out_path, mode="w+", dtype=np.float32, shape=(T, N2, 2))
    mm[:] = 0.0

    usecols = ["timestep", "node_idx", "water_level", "rainfall"]
    f = f"{base}/{ev}/2d_nodes_dynamic_all.csv"

    for chunk in pd.read_csv(f, usecols=usecols, chunksize=2_000_000):
        t = chunk["timestep"].to_numpy(np.int64)
        nid = chunk["node_idx"].to_numpy(np.int64)
        idx = np.fromiter((id2i_2d[int(x)] for x in nid), dtype=np.int64, count=nid.shape[0])

        mm[t, idx, 0] = chunk["water_level"].to_numpy(np.float32)
        mm[t, idx, 1] = chunk["rainfall"].to_numpy(np.float32)

    mm.flush()
    print("[cache] 2D", model_name, split, ev)

def cache_event_1d(model_name, split, ev, N1, id2i_1d):
    base = f"{DATA_ROOT}/{model_name}/{split}"
    cache_dir = ensure_cache_dir(model_name, split)
    out_dir = f"{cache_dir}/{ev}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/1d_dyn.dat"
    if os.path.exists(out_path):
        return

    mm = np.memmap(out_path, mode="w+", dtype=np.float32, shape=(T, N1, 1))
    mm[:] = 0.0

    usecols = ["timestep", "node_idx", "water_level"]
    f = f"{base}/{ev}/1d_nodes_dynamic_all.csv"

    for chunk in pd.read_csv(f, usecols=usecols, chunksize=2_000_000):
        t = chunk["timestep"].to_numpy(np.int64)
        nid = chunk["node_idx"].to_numpy(np.int64)
        idx = np.fromiter((id2i_1d[int(x)] for x in nid), dtype=np.int64, count=nid.shape[0])

        mm[t, idx, 0] = chunk["water_level"].to_numpy(np.float32)

    mm.flush()
    print("[cache] 1D", model_name, split, ev)

def open_mm_2d(model_name, split, ev, N2):
    cache_dir = ensure_cache_dir(model_name, split)
    path = f"{cache_dir}/{ev}/2d_dyn.dat"
    return np.memmap(path, mode="r", dtype=np.float32, shape=(T, N2, 2))

def open_mm_1d(model_name, split, ev, N1):
    cache_dir = ensure_cache_dir(model_name, split)
    path = f"{cache_dir}/{ev}/1d_dyn.dat"
    return np.memmap(path, mode="r", dtype=np.float32, shape=(T, N1, 1))

def cache_all(model_name, pack):
    for split in [SPLIT_TRAIN, SPLIT_TEST]:
        _, evs = list_events(model_name, split)
        for ev in evs:
            cache_event_2d(model_name, split, ev, pack["N2"], pack["id2i_2d"])
            cache_event_1d(model_name, split, ev, pack["N1"], pack["id2i_1d"])


In [7]:
# -----------------------------
# Coupled subgraph sampler
# -----------------------------
def sample_subgraph_coupled(pack,
                            seed_n2=SEED_N2, neigh_k2=NEIGH_K2,
                            seed_n1=SEED_N1, neigh_k1=NEIGH_K1,
                            expand_1d_hops=EXPAND_1D_HOPS):

    N1, N2 = pack["N1"], pack["N2"]

    # 2D sample
    src2 = pack["src2"]
    dst2 = pack["dst2"]
    src2_s = pack["src2_s"]
    dst2_s = pack["dst2_s"]
    indptr2 = pack["indptr2"]

    seed_n2 = min(seed_n2, N2)
    seeds2 = rng.randint(0, N2, size=seed_n2, dtype=np.int64)
    nodes2 = set(seeds2.tolist())

    for u in seeds2:
        a, b = indptr2[u], indptr2[u+1]
        if b <= a:
            continue
        nbrs = dst2_s[a:b]
        if nbrs.shape[0] > neigh_k2:
            take = rng.choice(nbrs.shape[0], size=neigh_k2, replace=False)
            nbrs = nbrs[take]
        for v in nbrs.tolist():
            nodes2.add(int(v))

    nodes2 = np.fromiter(nodes2, dtype=np.int64)
    nodes2.sort()

    mask2 = np.zeros(N2, dtype=np.bool_)
    mask2[nodes2] = True

    e_mask2 = mask2[src2] & mask2[dst2]
    e_idx2 = np.where(e_mask2)[0].astype(np.int64)

    re2 = np.full(N2, -1, dtype=np.int64)
    re2[nodes2] = np.arange(nodes2.shape[0], dtype=np.int64)
    src2_l = re2[src2[e_idx2]]
    dst2_l = re2[dst2[e_idx2]]

    # bring in 1D via links
    conn_src1 = pack["conn_src1"]
    conn_dst2 = pack["conn_dst2"]
    link_mask = mask2[conn_dst2]
    link_idx = np.where(link_mask)[0].astype(np.int64)

    nodes1 = set(conn_src1[link_idx].tolist())

    # expand 1D
    src1_s = pack["src1_s"]
    dst1_s = pack["dst1_s"]
    indptr1 = pack["indptr1"]

    for _ in range(expand_1d_hops):
        cur = np.fromiter(nodes1, dtype=np.int64)
        if cur.shape[0] == 0:
            break
        if cur.shape[0] > seed_n1:
            cur = rng.choice(cur, size=seed_n1, replace=False)
        for u in cur:
            a, b = indptr1[u], indptr1[u+1]
            if b <= a:
                continue
            nbrs = dst1_s[a:b]
            if nbrs.shape[0] > neigh_k1:
                take = rng.choice(nbrs.shape[0], size=neigh_k1, replace=False)
                nbrs = nbrs[take]
            for v in nbrs.tolist():
                nodes1.add(int(v))

    nodes1 = np.fromiter(nodes1, dtype=np.int64)
    nodes1.sort()

    mask1 = np.zeros(N1, dtype=np.bool_)
    mask1[nodes1] = True

    src1 = pack["src1"]
    dst1 = pack["dst1"]
    e_mask1 = mask1[src1] & mask1[dst1]
    e_idx1 = np.where(e_mask1)[0].astype(np.int64)

    re1 = np.full(N1, -1, dtype=np.int64)
    re1[nodes1] = np.arange(nodes1.shape[0], dtype=np.int64)
    src1_l = re1[src1[e_idx1]]
    dst1_l = re1[dst1[e_idx1]]

    # link edges within subgraph
    link_mask2 = mask1[conn_src1] & mask2[conn_dst2]
    link_idx2 = np.where(link_mask2)[0].astype(np.int64)

    l_src1_l = re1[conn_src1[link_idx2]]
    l_dst2_l = re2[conn_dst2[link_idx2]]

    return nodes1, nodes2, e_idx1, e_idx2, link_idx2, src1_l, dst1_l, src2_l, dst2_l, l_src1_l, l_dst2_l


In [8]:
# -----------------------------
# Coupled model: 1D + 2D + link messages
# -----------------------------
class CoupledMPNNGRU(nn.Module):
    def __init__(self, x_dim_1d, x_dim_2d, s_dim_1d, s_dim_2d, e_dim_1d, e_dim_2d, e_dim_link,
                 h=128, s_emb=64, e_emb=64):
        super().__init__()
        self.h = h

        self.s_proj_1d = nn.Linear(s_dim_1d, s_emb)
        self.s_proj_2d = nn.Linear(s_dim_2d, s_emb)

        self.e_proj_1d = nn.Linear(e_dim_1d, e_emb)
        self.e_proj_2d = nn.Linear(e_dim_2d, e_emb)
        self.e_proj_lk = nn.Linear(e_dim_link, e_emb)

        self.x_proj_1d = nn.Linear(x_dim_1d + s_emb, h)
        self.x_proj_2d = nn.Linear(x_dim_2d + s_emb, h)

        # 1D edge messages
        self.msg_1d = nn.Sequential(nn.Linear(h + e_emb, h), nn.ReLU(), nn.Linear(h, h))
        self.gate_1d = nn.Sequential(nn.Linear(e_emb, h), nn.Sigmoid())

        # 2D edge messages
        self.msg_2d = nn.Sequential(nn.Linear(h + e_emb, h), nn.ReLU(), nn.Linear(h, h))
        self.gate_2d = nn.Sequential(nn.Linear(e_emb, h), nn.Sigmoid())

        # link messages: 1D->2D and 2D->1D
        self.msg_12 = nn.Sequential(nn.Linear(h + e_emb, h), nn.ReLU(), nn.Linear(h, h))
        self.gate_12 = nn.Sequential(nn.Linear(e_emb, h), nn.Sigmoid())

        self.msg_21 = nn.Sequential(nn.Linear(h + e_emb, h), nn.ReLU(), nn.Linear(h, h))
        self.gate_21 = nn.Sequential(nn.Linear(e_emb, h), nn.Sigmoid())

        self.gru_1d = nn.GRUCell(h + h, h)
        self.gru_2d = nn.GRUCell(h + h, h)
        self.norm_1d = nn.LayerNorm(h)
        self.norm_2d = nn.LayerNorm(h)

        self.readout_1d = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
        self.readout_2d = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))

        self.D1 = 1.0
        self.D2 = 1.0

    def set_delta_scales(self, D1, D2):
        self.D1 = float(D1)
        self.D2 = float(D2)

    def _squash(self, d, D):
        return D * torch.tanh(d / D)

    def step(self, x1_t, x2_t,
             s1_emb, s2_emb,
             src1, dst1, e1_emb,
             src2, dst2, e2_emb,
             l_src1, l_dst2, el_emb,
             h1, h2, rounds=2):

        xh1 = self.x_proj_1d(torch.cat([x1_t, s1_emb], dim=1))
        xh2 = self.x_proj_2d(torch.cat([x2_t, s2_emb], dim=1))

        N1 = xh1.shape[0]
        N2 = xh2.shape[0]

        for _ in range(rounds):
            agg1 = torch.zeros(N1, self.h, device=h1.device, dtype=xh1.dtype)
            agg2 = torch.zeros(N2, self.h, device=h2.device, dtype=xh2.dtype)
            deg1 = torch.zeros(N1, device=h1.device, dtype=xh1.dtype)
            deg2 = torch.zeros(N2, device=h2.device, dtype=xh2.dtype)

            # 1D edges
            m1 = self.msg_1d(torch.cat([h1[src1], e1_emb], dim=1)) * self.gate_1d(e1_emb)
            agg1.index_add_(0, dst1, m1)
            deg1.index_add_(0, dst1, torch.ones_like(dst1, dtype=deg1.dtype))

            # 2D edges
            m2 = self.msg_2d(torch.cat([h2[src2], e2_emb], dim=1)) * self.gate_2d(e2_emb)
            agg2.index_add_(0, dst2, m2)
            deg2.index_add_(0, dst2, torch.ones_like(dst2, dtype=deg2.dtype))

            # link 1D -> 2D
            m12 = self.msg_12(torch.cat([h1[l_src1], el_emb], dim=1)) * self.gate_12(el_emb)
            agg2.index_add_(0, l_dst2, m12)
            deg2.index_add_(0, l_dst2, torch.ones_like(l_dst2, dtype=deg2.dtype))

            # link 2D -> 1D
            m21 = self.msg_21(torch.cat([h2[l_dst2], el_emb], dim=1)) * self.gate_21(el_emb)
            agg1.index_add_(0, l_src1, m21)
            deg1.index_add_(0, l_src1, torch.ones_like(l_src1, dtype=deg1.dtype))

            agg1 = agg1 / deg1.clamp_min(1.0).unsqueeze(1)
            agg2 = agg2 / deg2.clamp_min(1.0).unsqueeze(1)

            h1 = self.norm_1d(self.gru_1d(torch.cat([xh1, agg1], dim=1), h1))
            h2 = self.norm_2d(self.gru_2d(torch.cat([xh2, agg2], dim=1), h2))

        d1 = self.readout_1d(h1).squeeze(1)
        d2 = self.readout_2d(h2).squeeze(1)

        d1 = self._squash(d1, self.D1)
        d2 = self._squash(d2, self.D2)

        return h1, h2, d1, d2


In [9]:
# -----------------------------
# Optional: load 2D-only weights into the coupled model's 2D branch
#   Put checkpoint here: checkpoints/{ModelName}_2d_ckpt.pt
# -----------------------------
def load_pretrained_2d_into_coupled(model_name, coupled_model):
    ckpt_path = f"{CKPT_DIR}/{model_name}_2d_ckpt.pt"
    if not os.path.exists(ckpt_path):
        print("[pretrain] no 2D ckpt found for", model_name, "at", ckpt_path)
        return

    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt["model"] if "model" in ckpt else ckpt

    with torch.no_grad():
        # common keys from your 2D-only notebook: s_proj, e_proj, x_proj, msg_mlp, edge_gate, gru, norm, readout
        coupled_model.s_proj_2d.weight.copy_(sd["s_proj.weight"])
        coupled_model.s_proj_2d.bias.copy_(sd["s_proj.bias"])

        coupled_model.e_proj_2d.weight.copy_(sd["e_proj.weight"])
        coupled_model.e_proj_2d.bias.copy_(sd["e_proj.bias"])

        coupled_model.x_proj_2d.weight.copy_(sd["x_proj.weight"])
        coupled_model.x_proj_2d.bias.copy_(sd["x_proj.bias"])

        coupled_model.msg_2d[0].weight.copy_(sd["msg_mlp.0.weight"])
        coupled_model.msg_2d[0].bias.copy_(sd["msg_mlp.0.bias"])
        coupled_model.msg_2d[2].weight.copy_(sd["msg_mlp.2.weight"])
        coupled_model.msg_2d[2].bias.copy_(sd["msg_mlp.2.bias"])

        coupled_model.gate_2d[0].weight.copy_(sd["edge_gate.0.weight"])
        coupled_model.gate_2d[0].bias.copy_(sd["edge_gate.0.bias"])

        coupled_model.gru_2d.weight_ih.copy_(sd["gru.weight_ih"])
        coupled_model.gru_2d.weight_hh.copy_(sd["gru.weight_hh"])
        coupled_model.gru_2d.bias_ih.copy_(sd["gru.bias_ih"])
        coupled_model.gru_2d.bias_hh.copy_(sd["gru.bias_hh"])

        coupled_model.norm_2d.weight.copy_(sd["norm.weight"])
        coupled_model.norm_2d.bias.copy_(sd["norm.bias"])

        coupled_model.readout_2d[0].weight.copy_(sd["readout.0.weight"])
        coupled_model.readout_2d[0].bias.copy_(sd["readout.0.bias"])
        coupled_model.readout_2d[2].weight.copy_(sd["readout.2.weight"])
        coupled_model.readout_2d[2].bias.copy_(sd["readout.2.bias"])

    print("[pretrain] loaded 2D weights for", model_name, "from", ckpt_path)


In [10]:
# -----------------------------
# Estimate delta scales (so you don't cap floods into irrelevance)
# -----------------------------
def estimate_delta_scales(model_name, pack, event_dirs, q=0.995):
    N1, N2 = pack["N1"], pack["N2"]
    d1 = []
    d2 = []

    for _ in range(10):
        ev = event_dirs[rng.randint(0, len(event_dirs))]
        mm1 = open_mm_1d(model_name, SPLIT_TRAIN, ev, N1)
        mm2 = open_mm_2d(model_name, SPLIT_TRAIN, ev, N2)

        ts = rng.randint(0, T-2, size=500, dtype=np.int64)
        n1 = min(N1, 1024)
        n2 = min(N2, 2048)
        i1 = rng.randint(0, N1, size=n1, dtype=np.int64)
        i2 = rng.randint(0, N2, size=n2, dtype=np.int64)

        w1_t = mm1[ts, :][:, i1, 0]
        w1_n = mm1[ts+1, :][:, i1, 0]
        w2_t = mm2[ts, :][:, i2, 0]
        w2_n = mm2[ts+1, :][:, i2, 0]

        d1.append(np.abs(w1_n - w1_t).reshape(-1))
        d2.append(np.abs(w2_n - w2_t).reshape(-1))

    d1 = np.concatenate(d1)
    d2 = np.concatenate(d2)
    D1 = float(np.quantile(d1, q) + 1e-6)
    D2 = float(np.quantile(d2, q) + 1e-6)
    return D1, D2


In [11]:
# -----------------------------
# Training loop
# -----------------------------
def train_coupled(model_name, pack, train_events):

    N1, N2 = pack["N1"], pack["N2"]

    model = CoupledMPNNGRU(
        x_dim_1d=4,   # [wl1, rain_link, rsum3_link, rsum10_link]
        x_dim_2d=4,   # [wl2, rain2, rsum3, rsum10]
        s_dim_1d=pack["node_static_1d"].shape[1],
        s_dim_2d=pack["node_static_2d"].shape[1],
        e_dim_1d=pack["edge_attr_1d"].shape[1],
        e_dim_2d=pack["edge_attr_2d"].shape[1],
        e_dim_link=pack["link_attr"].shape[1],
        h=128,
        s_emb=64,
        e_emb=64
    ).to(device)

    if USE_PRETRAIN_2D:
        load_pretrained_2d_into_coupled(model_name, model)

    D1, D2 = estimate_delta_scales(model_name, pack, train_events, q=0.995)
    model.set_delta_scales(D1, D2)
    print(f"[scales] {model_name} D1={D1:.6f} D2={D2:.6f}")

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler(device.type, enabled=USE_SCALER)

    ckpt_path = f"{CKPT_DIR}/{model_name}_coupled_ckpt.pt"
    start_step = 0
    if os.path.exists(ckpt_path):
        start_step = load_ckpt(ckpt_path, model, opt)
        print("resume", model_name, "from step", start_step)

    model.train()
    global_step = start_step

    for it in range(STEPS):
        ev = train_events[rng.randint(0, len(train_events))]
        mm1 = open_mm_1d(model_name, SPLIT_TRAIN, ev, N1)
        mm2 = open_mm_2d(model_name, SPLIT_TRAIN, ev, N2)

        t0 = rng.randint(0, T - (L + 1))

        (nodes1, nodes2,
         e_idx1, e_idx2, link_idx,
         src1_l, dst1_l, src2_l, dst2_l,
         l_src1_l, l_dst2_l) = sample_subgraph_coupled(pack)

        # dynamic slices
        X2 = mm2[t0:t0+L+1, nodes2, :]  # (L+1, n2, 2)
        wl2 = torch.tensor(X2[:, :, 0], device=device)
        rain2 = torch.tensor(X2[:, :, 1], device=device)

        X1 = mm1[t0:t0+L+1, nodes1, :]  # (L+1, n1, 1)
        wl1 = torch.tensor(X1[:, :, 0], device=device)

        # rainfall sums for 2D nodes from true rainfall history
        start = max(0, t0 - 9)
        rbuf = torch.tensor(mm2[start:t0+L+1, nodes2, 1], device=device)
        cs = torch.cumsum(rbuf, dim=0)
        cs = torch.cat([torch.zeros(1, cs.shape[1], device=device, dtype=cs.dtype), cs], dim=0)
        i0 = t0 - start
        idx = torch.arange(i0, i0 + (L + 1), device=device)
        rain_sum3_2d  = cs[idx + 1] - cs[torch.clamp(idx + 1 - 3, min=0)]
        rain_sum10_2d = cs[idx + 1] - cs[torch.clamp(idx + 1 - 10, min=0)]

        # static tensors
        s1 = pack["node_static_1d"][nodes1]
        s2 = pack["node_static_2d"][nodes2]
        e1 = pack["edge_attr_1d"][e_idx1]
        e2 = pack["edge_attr_2d"][e_idx2]
        el = pack["link_attr"][link_idx]

        src1 = torch.tensor(src1_l, dtype=torch.long, device=device)
        dst1 = torch.tensor(dst1_l, dtype=torch.long, device=device)
        src2 = torch.tensor(src2_l, dtype=torch.long, device=device)
        dst2 = torch.tensor(dst2_l, dtype=torch.long, device=device)

        l_src1 = torch.tensor(l_src1_l, dtype=torch.long, device=device)
        l_dst2 = torch.tensor(l_dst2_l, dtype=torch.long, device=device)

        # embeddings
        s1_emb = model.s_proj_1d(s1)
        s2_emb = model.s_proj_2d(s2)
        e1_emb = model.e_proj_1d(e1)
        e2_emb = model.e_proj_2d(e2)
        el_emb = model.e_proj_lk(el)

        # hidden states
        h1 = torch.zeros(nodes1.shape[0], model.h, device=device)
        h2 = torch.zeros(nodes2.shape[0], model.h, device=device)

        p_ss = ss_prob(global_step)

        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = 0.0
            n_loss = 0

            wl1_in = wl1[0]
            wl2_in = wl2[0]

            for t in range(0, L):
                # mean rainfall features onto 1D nodes via link edges
                r1 = torch.zeros(nodes1.shape[0], device=device, dtype=rain2.dtype)
                r3 = torch.zeros_like(r1)
                r10 = torch.zeros_like(r1)
                cnt = torch.zeros_like(r1)

                r_link   = rain2[t, l_dst2]
                r3_link  = rain_sum3_2d[t, l_dst2]
                r10_link = rain_sum10_2d[t, l_dst2]

                r1.index_add_(0, l_src1, r_link)
                r3.index_add_(0, l_src1, r3_link)
                r10.index_add_(0, l_src1, r10_link)
                cnt.index_add_(0, l_src1, torch.ones_like(r_link))

                cnt = cnt.clamp_min(1.0)
                r1  = r1 / cnt
                r3  = r3 / cnt
                r10 = r10 / cnt

                x1_t = torch.stack([wl1_in, r1, r3, r10], dim=1)
                x2_t = torch.stack([wl2_in, rain2[t], rain_sum3_2d[t], rain_sum10_2d[t]], dim=1)

                h1, h2, d1, d2 = model.step(
                    x1_t, x2_t,
                    s1_emb, s2_emb,
                    src1, dst1, e1_emb,
                    src2, dst2, e2_emb,
                    l_src1, l_dst2, el_emb,
                    h1, h2,
                    rounds=ROUNDS
                )

                wl1_pred_next = wl1_in + d1
                wl2_pred_next = wl2_in + d2

                wl1_true_next = wl1[t+1]
                wl2_true_next = wl2[t+1]

                if (t0 + t) >= WARMUP:
                    loss = loss + (ALPHA_1D * F.mse_loss(wl1_pred_next, wl1_true_next) +
                                   F.mse_loss(wl2_pred_next, wl2_true_next))
                    n_loss += 1

                if (t0 + t + 1) >= WARMUP:
                    wl1_in = p_ss * wl1_pred_next + (1.0 - p_ss) * wl1_true_next
                    wl2_in = p_ss * wl2_pred_next + (1.0 - p_ss) * wl2_true_next
                else:
                    wl1_in = wl1_true_next
                    wl2_in = wl2_true_next

            loss = loss / max(1, n_loss)

        if not torch.isfinite(loss):
            print("[bad] non-finite loss:", loss.item(), "ev=", ev, "t0=", t0)
            save_ckpt(ckpt_path, model, opt, global_step)
            break

        if USE_SCALER:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step()

        global_step += 1

        if (it + 1) % 100 == 0:
            print(f"[{model_name}] it={it+1} loss={float(loss.item()):.6f} ss={p_ss:.3f} ev={ev} t0={t0} n1={nodes1.shape[0]} n2={nodes2.shape[0]} e1={len(e_idx1)} e2={len(e_idx2)} link={len(link_idx)}")

        if (it + 1) % 500 == 0:
            save_ckpt(ckpt_path, model, opt, global_step)

    save_ckpt(ckpt_path, model, opt, global_step)
    print("[done] saved", ckpt_path)


In [12]:
# -----------------------------
# Run: build packs, sanity checks, cache, train
# -----------------------------
systems = {}

for model_name in MODELS:
    print("\n=== setup", model_name, "===")

    (df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
     df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static) = load_static(model_name)

    pack = build_pack(df_1d_edge_index, df_1d_edges_static, df_1d_nodes_static, df_1d2d_connections,
                      df_2d_edge_index, df_2d_edges_static, df_2d_nodes_static)

    systems[model_name] = {"pack": pack}

    print("Sanity checks (quick):")
    sanity_check_1d_dynamic(model_name, SPLIT_TRAIN, pack, n_events=2)
    sanity_check_2d_dynamic(model_name, SPLIT_TRAIN, pack, n_events=1)

    print("Caching memmaps (train+test):")
    cache_all(model_name, pack)

    _, train_events = list_events(model_name, SPLIT_TRAIN)
    systems[model_name]["train_events"] = train_events

for model_name in MODELS:
    print("\n=== train coupled", model_name, "===")
    pack = systems[model_name]["pack"]
    train_events = systems[model_name]["train_events"]
    train_coupled(model_name, pack, train_events)



=== setup Model_1 ===
Sanity checks (quick):
Model_1 train event_1 static_nodes= 17 dyn_nodes= 17 extra= 0 missing= 0
Model_1 train event_10 static_nodes= 17 dyn_nodes= 17 extra= 0 missing= 0
Model_1 train event_1 static_nodes= 3716 dyn_nodes= 3716 extra= 0 missing= 0
Caching memmaps (train+test):
[cache] 2D Model_1 train event_1
[cache] 1D Model_1 train event_1
[cache] 2D Model_1 train event_10
[cache] 1D Model_1 train event_10
[cache] 2D Model_1 train event_11
[cache] 1D Model_1 train event_11
[cache] 2D Model_1 train event_12
[cache] 1D Model_1 train event_12
[cache] 2D Model_1 train event_13
[cache] 1D Model_1 train event_13
[cache] 2D Model_1 train event_14
[cache] 1D Model_1 train event_14
[cache] 2D Model_1 train event_15
[cache] 1D Model_1 train event_15
[cache] 2D Model_1 train event_16
[cache] 1D Model_1 train event_16
[cache] 2D Model_1 train event_17
[cache] 1D Model_1 train event_17
[cache] 2D Model_1 train event_19
[cache] 1D Model_1 train event_19
[cache] 2D Model_1 tra